# **Data engineering**

In [1]:
"""
monthly_pipeline_transformers.py

Transformers compatibles con sklearn Pipeline para:
 - preprocessar df_stores (StorePreprocessor)
 - preprocessar df_orders (OrdersPreprocessor)
 - construir monthly_master a partir de los dos (MonthlyMasterBuilder)

Uso:
 1) Instanciar los transformadores y armar un Pipeline si querés (ver ejemplo al final).
 2) MonthlyMasterBuilder.transform() espera un dict: {'orders': df_orders_preproc, 'stores': df_stores_preproc}

NOTA: sklearn.Pipeline normalmente pasa un único X entre transformadores. Aquí proponemos usar transformadores
que aceptan/retornan DataFrames (o dicts) para facilitar el flujo pandas-native. Si preferís una integración estricta con
sklearn (matrices numpy), hay que adaptar a FunctionTransformer que empaqueta/desempaqueta.

"""
from typing import Optional, Dict
import pandas as pd
import numpy as np
import unicodedata
from sklearn.base import BaseEstimator, TransformerMixin

In [20]:
class StorePreprocessor(BaseEstimator, TransformerMixin):
    """Preprocesa df_stores y devuelve un DataFrame limpio listo para mergear.

    Transformaciones realizadas (replican tu EDA):
    - Agrupa por store_id y aplica bfill+take first
    - Filtra status == 'active' (si existe)
    - Rellena brand_id faltantes con ids únicos incrementales
    - Normaliza created_date a UTC y lo deja "naive" (sin tz) con .normalize()
    - Filtra segmento (no nulo y distinto de '0')
    - Mapea logistic_type -> in-house_delivery (3P->0, 2P->1)
    - Normaliza barrio (mapeo de variantes + lower sin tildes) y crea columna ciudad mediante mapping
    - Convierte cooking_time a Int64
    - Convierte tiene_otra_app a Int64 (Sí->1, No->0)
    """

    def __init__(self, barrio_normalization_dict: dict = None):
        # dict por defecto (podés extenderlo desde afuera)
        default_norm = {
            'Agronomia': 'Agronomia', 'Agronomía': 'Agronomia',
            'Monserrat': 'Monserrat', 'Montserrat': 'Monserrat',
            'Nunez': 'Nunez', 'Núñez': 'Nunez',
            'Villa General Mitre': 'Villa General Mitre', 'Villa Gral. Mitre': 'Villa General Mitre',
            'Villa Ortuzar': 'Villa Ortuzar', 'Villa Ortúzar': 'Villa Ortuzar',
            'Villa Pueyrredon': 'Villa Pueyrredon', 'Villa Pueyrredón': 'Villa Pueyrredon',
            'Villa del Parque': 'Villa Del Parque', 'Vélez Sársfield': 'Velez Sarsfield',
            'Velez Sarsfield': 'Velez Sarsfield',
            'Santa Rita': 'Villa Santa Rita', 'Villa Santa Rita': 'Villa Santa Rita',
            'Paternal': 'La Paternal', 'La Paternal': 'La Paternal', 'Versailles': 'Versalles',
        }
        self.barrio_normalization_dict = barrio_normalization_dict or default_norm
        # construir mapa con keys normalizadas (sin tildes y lower) para lookup robusto
        self._barrio_map = {
            self._lower_strip_accents(k): self._lower_strip_accents(v)
            for k, v in self.barrio_normalization_dict.items()
        }

        # --- listas de barrios usadas por _assign_city ---
        # Aquí definí las listas originales (tal como las tenías). Las normalizo y guardo como sets
        barrios_caba = [
            'agronomia', 'almagro', 'balvanera', 'barracas', 'belgrano', 'boedo', 'caballito', 'chacarita',
            'coghlan', 'colegiales', 'constitucion', 'flores', 'floresta', 'la boca', 'la paternal', 'liniers',
            'mataderos', 'monserrat', 'monte castro', 'nueva pompeya', 'nunez', 'palermo', 'parque avellaneda',
            'parque chacabuco', 'parque chas', 'parque patricios', 'puerto madero', 'recoleta', 'retiro', 'saavedra',
            'san cristobal', 'san nicolas', 'san telmo', 'velez sarsfield', 'versalles', 'villa crespo',
            'villa del parque', 'villa devoto', 'villa general mitre', 'villa lugano', 'villa luro',
            'villa ortuzar', 'villa pueyrredon', 'villa real', 'villa riachuelo', 'villa santa rita',
            'villa soldati', 'villa urquiza'
        ]

        barrios_gba = [
            'acassuso', 'avellaneda', 'beccar', 'boulogne sur mer', 'carapachay', 'ciudadela', 'crucesita',
            'florida', 'florida oeste', 'gerli', 'haedo', 'la lucila', 'lomas del mirador',
            'martinez', 'martinez oeste', 'moron', 'munro', 'olivos', 'pineyro', 'ramos mejia', 'san isidro', 'san justo',
            'san martin centro', 'vicente lopez', 'villa adelina', 'villa ballester', 'villa bonich',
            'villa luzuriaga', 'villa lynch', 'villa maipu', 'villa martelli', 'villa san andres',
            'villa sarmiento', 'garin'
        ]

        barrios_cordoba = [
            'alberdi', 'alta cordoba', 'altamira', 'alto alberdi', 'alto general paz', 'altos de velez sarsfield',
            'ameghino norte', 'ampliacion jardin espinosa', 'ampliacion kennedy', 'ampliacion pueyrredon', 'avenida',
            'bialet masse', 'california', 'carola lorenzini', 'caseros', 'centro', 'cerro de las rosas', 'cofico', 'colon',
            'crisol norte', 'crisol sud', 'ducasse', 'el trebol', 'empalme', 'ferrer', 'general bustos', 'general paz',
            'general pueyrredon', 'guemes', 'independencia', 'ipona', 'jardin', 'jardin del sud', 'jardin espinosa',
            'jardin hipodromo', 'jardines del jockey', 'jockey club', 'jose ignacio diaz cuarta seccion', 'juan xxiii',
            'juniors', 'kennedy', 'lamadrid', 'las flores', 'los naranjos', 'los olmos', 'los platanos', 'maipu primera seccion',
            'manantiales', 'mariano balcarce', 'nueva cordoba', 'obrero', 'observatorio', 'panamericano', 'parque atlantica',
            'parque capital', 'parque horizonte', 'parque san vicente', 'parque sarmiento', 'paso de los andes', 'poeta lugones',
            'providencia', 'puente blanco', 'quebrada de las rosas', 'quinta santa ana', 'residencial olivos', 'residencial san carlos',
            'rivadavia', 'rosedal', 'san cayetano', 'san daniel', 'san fernando', 'san martin', 'san rafael', 'san vicente', 'smata',
            'teniente benjamin matienzo', 'villa revol', 'villa san carlos', 'villa silvano funes', 'yapeyu'
        ]

        barrios_rosario = ['centro', 'las malvinas', 'lisandro de la torre', 'luis agote']

        # normalizo y guardo como sets para busquedas O(1)
        self._barrios_caba = {self._lower_strip_accents(x) for x in barrios_caba}
        self._barrios_gba = {self._lower_strip_accents(x) for x in barrios_gba}
        self._barrios_cordoba = {self._lower_strip_accents(x) for x in barrios_cordoba}
        self._barrios_rosario = {self._lower_strip_accents(x) for x in barrios_rosario}

    def normalize_barrio(self, barrio):
        """Normaliza un barrio: mapea variaciones conocidas y devuelve en forma 'lower no-accent'."""
        if pd.isnull(barrio):
            return barrio
        key = str(barrio).strip()
        key_norm = self._lower_strip_accents(key)
        mapped = self._barrio_map.get(key_norm, key_norm)
        return mapped

    # -------------------- helpers --------------------
    @staticmethod
    def _lower_strip_accents(s: Optional[str]) -> Optional[str]:
        if pd.isnull(s):
            return s
        s = str(s).lower()
        s = unicodedata.normalize('NFKD', s).encode('ASCII', 'ignore').decode('utf-8')
        return s

    def _assign_city(self, barrio: str) -> Optional[str]:
        """Asigna ciudad a partir de barrio ya normalizado (lower, sin tildes)."""
        if not isinstance(barrio, str):
            return None
        barrio_norm = barrio  # ya debería venir normalizado por normalize_barrio
        if barrio_norm in self._barrios_caba:
            return 'CABA'
        if barrio_norm in self._barrios_gba:
            return 'GBA'
        if barrio_norm in self._barrios_cordoba:
            return 'Córdoba'
        if barrio_norm in self._barrios_rosario:
            return 'Rosario'
        if 'la plata' in barrio_norm:
            return 'La Plata'
        return None

    # -------------------- fit/transform --------------------
    def fit(self, X: pd.DataFrame, y=None):
        # no-op: este transformer no aprende parámetros
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()

        # 1) intentar agrupar por store_id y bfill+take first
        if 'store_id' in df.columns:
            try:
                df = df.groupby('store_id', sort=False).apply(lambda g: g.bfill().iloc[0]).reset_index(drop=True)
            except Exception:
                df = df.copy()

        # 2) status active
        if 'status' in df.columns:
            df = df[df['status'] == 'active'].copy()
            df.drop(columns=['status'], inplace=True, errors='ignore')

        # 3) brand_id
        if 'brand_id' in df.columns:
            # forzar numeric para calcular max si es posible
            brand_numeric = pd.to_numeric(df['brand_id'], errors='coerce')
            if brand_numeric.dropna().shape[0] > 0:
                max_brand = int(brand_numeric.dropna().max())
                contador = max_brand + 1
            else:
                contador = 1
            mask_na = df['brand_id'].isna()
            if mask_na.any():
                df.loc[mask_na, 'brand_id'] = range(contador, contador + mask_na.sum())
            df['brand_id'] = df['brand_id'].astype(int)

        # 4) created_date parse
        if 'created_date' in df.columns:
            df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce', utc=True).dt.tz_convert("UTC").dt.tz_localize(None).dt.normalize()

        # 5) segmento filter
        if 'segmento' in df.columns:
            df = df[df['segmento'].notna() & (df['segmento'] != "0")].copy()

        # 6) logistic_type -> in-house_delivery
        if 'logistic_type' in df.columns:
            df.rename(columns={'logistic_type': 'in-house_delivery'}, inplace=True)
            df['in-house_delivery'] = df['in-house_delivery'].replace({'3P': '0', '2P': '1'}).astype(int)

        # 7) barrio -> aplicar normalización robusta y asignar ciudad
        if 'barrio' in df.columns:
            # aplicar la normalización robusta (respeta NaN)
            df['barrio'] = df['barrio'].apply(lambda x: self.normalize_barrio(x) if pd.notna(x) else x).astype(str)
            # ahora assign_city usa barrio ya normalizado (lower y sin tildes)
            df['ciudad'] = df['barrio'].apply(self._assign_city)

        # 8) cooking_time
        if 'cooking_time' in df.columns:
            df['cooking_time'] = df['cooking_time'].astype(str).str.replace('min', '', regex=False).str.strip()
            df['cooking_time'] = pd.to_numeric(df['cooking_time'], errors='coerce').astype('Int64')

        # 9) tiene_otra_app
        if 'tiene_otra_app' in df.columns:
            df['tiene_otra_app'] = df['tiene_otra_app'].replace({'Sí': '1', 'No': '0'}).astype('Int64')

        return df.reset_index(drop=True)

In [3]:
class OrdersPreprocessor(BaseEstimator, TransformerMixin):
    """Preprocesa df_orders; devuelve DataFrame.

    - Convierte orders_date a datetime
    - Crea flags si_ios, si_meli, si_other a partir de app_origin
    - Convierte columnas numéricas (tpn, gmv, avg_shipping_cost, markdown)
    - Drop tpv si existe
    """

    def __init__(self):
        pass

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()
        if 'orders_date' in df.columns:
            df['orders_date'] = pd.to_datetime(df['orders_date'], errors='coerce')
        elif {'year', 'month'}.issubset(df.columns):
            df['orders_date'] = pd.to_datetime(df['year'].astype(int).astype(str) + '-' + df['month'].astype(int).astype(str) + '-01')

        if 'app_origin' in df.columns:
            df['si_ios'] = df['app_origin'].str.contains('IOS', case=False, na=False).astype(int)
            df['si_meli'] = df['app_origin'].str.contains('MELI', case=False, na=False).astype(int)
            df['si_other'] = df['app_origin'].str.contains('OTHER', case=False, na=False).astype(int)

        if 'tpv' in df.columns:
            df = df.drop(columns=['tpv'], errors='ignore')

        for c in ['tpn', 'gmv', 'avg_shipping_cost', 'markdown']:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors='coerce')

        return df.reset_index(drop=True)

In [4]:
class MonthlyMasterBuilder(BaseEstimator, TransformerMixin):
    """Construye monthly_master a partir de {'orders': df_orders, 'stores': df_stores}.

    EXPECTATIVA de input para transform(X):
        X debe ser un dict con llaves 'orders' y 'stores' conteniendo DataFrames ya preprocesados.

    Salida: DataFrame monthly_master con las columnas calculadas en tu EDA (tpn, gmv, churn, stage, etc.)
    """

    def __init__(self):
        pass

    @staticmethod
    def _consecutive_ones_from_churn(s: pd.Series) -> pd.Series:
        mask_nan = s.isna()
        s0 = s.fillna(0).astype(int)
        groups = (s0 != s0.shift()).cumsum()
        streak = s0.groupby(groups).cumcount() + 1
        streak = (streak * s0).astype(float)
        streak[mask_nan] = np.nan
        return streak

    def fit(self, X: Dict[str, pd.DataFrame], y=None):
        # no-op
        return self

    def transform(self, X: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        if not isinstance(X, dict) or 'orders' not in X or 'stores' not in X:
            raise ValueError("MonthlyMasterBuilder.transform espera un dict {'orders': df_orders, 'stores': df_stores}")

        orders = X['orders'].copy()
        stores = X['stores'].copy()

        if 'orders_date' not in orders.columns:
            raise ValueError("df_orders debe tener columna 'orders_date' o (year,month)")
        orders['orders_date'] = pd.to_datetime(orders['orders_date'], errors='coerce')

        # Aggregación mensual
        orders['period_month'] = orders['orders_date'].dt.to_period('M')

        agg_args = {
            'tpn': pd.NamedAgg(column='tpn', aggfunc='sum'),
            'gmv': pd.NamedAgg(column='gmv', aggfunc='sum') if 'gmv' in orders.columns else pd.NamedAgg(column='tpn', aggfunc='sum'),
            'days_with_orders': pd.NamedAgg(column='orders_date', aggfunc=lambda s: s.dt.date.nunique()),
            'markdown_mean': pd.NamedAgg(column='markdown', aggfunc='mean'),
            'avg_shipping_cost': pd.NamedAgg(column='avg_shipping_cost', aggfunc='mean'),
        }
        for col in ['si_ios','si_meli','si_other','free_shipping']:
            if col in orders.columns:
                agg_args[f'{col}_prop'] = pd.NamedAgg(column=col, aggfunc=lambda s: s.eq(1).mean())

        orders_monthly = (
            orders
            .groupby(['store_id','period_month'], observed=True)
            .agg(**agg_args)
            .reset_index()
        )

        # último periodo observado
        if orders_monthly.empty:
            last_period = pd.Period(pd.Timestamp.now(), freq='M')
        else:
            last_period = orders_monthly['period_month'].max()
            if pd.isna(last_period):
                last_period = pd.Period(pd.Timestamp.now(), freq='M')

        # grid mensual desde created_date hasta last_period
        stores_iter = stores[['store_id','created_date']].drop_duplicates('store_id')
        parts = []
        for row in stores_iter.itertuples(index=False):
            sid = row.store_id
            created = row.created_date if hasattr(row, 'created_date') else None
            if pd.notna(created):
                start = pd.Period(created, freq='M')
            else:
                first_order_series = orders_monthly.loc[orders_monthly['store_id']==sid, 'period_month']
                if not first_order_series.empty:
                    start = first_order_series.min()
                else:
                    continue
            if start > last_period:
                continue
            idx = pd.period_range(start, last_period, freq='M')
            tmp = pd.DataFrame({'store_id': sid, 'period_month': idx})
            parts.append(tmp)

        if not parts:
            raise ValueError("No se generaron periodos: revisá created_date y orders_monthly.")
        grid_monthly = pd.concat(parts, ignore_index=True).sort_values(['store_id','period_month']).reset_index(drop=True)

        monthly_full = grid_monthly.merge(orders_monthly, on=['store_id','period_month'], how='left')

        monthly_full['tpn'] = monthly_full['tpn'].fillna(0).astype(float)
        monthly_full['gmv'] = monthly_full['gmv'].fillna(0).astype(float)
        monthly_full['days_with_orders'] = monthly_full['days_with_orders'].fillna(0).astype(int)

        # churn logic
        monthly_full = monthly_full.sort_values(['store_id','period_month']).reset_index(drop=True)
        grp = monthly_full.groupby('store_id')['tpn']

        ever_pos_cum = grp.transform(lambda x: x.gt(0).cumsum())
        had_positive_before = grp.transform(lambda x: x.gt(0).cumsum().shift(fill_value=0).gt(0))

        monthly_full['churn'] = np.where(
            ever_pos_cum == 0,
            np.nan,
            ((monthly_full['tpn'] == 0) & had_positive_before).astype(float)
        )

        monthly_full['churn_streak'] = monthly_full.groupby('store_id')['churn'].transform(self._consecutive_ones_from_churn)

        monthly_full['churn_2m_consec'] = np.where(
            monthly_full['churn_streak'].isna(), np.nan, (monthly_full['churn_streak'] >= 2).astype(int)
        )
        monthly_full['churn_4m_consec_or_more'] = np.where(
            monthly_full['churn_streak'].isna(), np.nan, (monthly_full['churn_streak'] >= 4).astype(int)
        )

        monthly_full['churn_unique'] = np.where(
            monthly_full['churn_streak'].isna(), np.nan,
            ((monthly_full['churn'] == 1) & (monthly_full['churn_streak'] == 1)).astype(int)
        )

        # primer periodo con venta y prev_churn
        first_pos_map = monthly_full.loc[monthly_full['tpn'] > 0].groupby('store_id')['period_month'].min()
        monthly_full['first_positive_period'] = monthly_full['store_id'].map(first_pos_map)
        monthly_full['prev_churn'] = monthly_full.groupby('store_id')['churn'].shift(1)

        # stage
        monthly_full['stage'] = np.nan
        mask_before_first_sale = monthly_full['churn'].isna()
        monthly_full.loc[mask_before_first_sale, 'stage'] = np.nan

        mask_churn = (monthly_full['tpn'] == 0) & monthly_full['churn_streak'].notna() & (monthly_full['churn_streak'] > 0)
        monthly_full.loc[mask_churn & (monthly_full['churn_streak'] >= 4), 'stage'] = 'Churn > 3M'
        monthly_full.loc[mask_churn & (monthly_full['churn_streak'] == 1), 'stage'] = 'Churn'
        monthly_full.loc[mask_churn & (monthly_full['churn_streak'] > 1) & (monthly_full['churn_streak'] <= 3), 'stage'] = 'Churn <= 3M'

        mask_sale = (monthly_full['tpn'] > 0) & monthly_full['churn'].notna()
        mask_first_positive = mask_sale & (monthly_full['period_month'] == monthly_full['first_positive_period'])
        monthly_full.loc[mask_first_positive, 'stage'] = 'New'
        mask_reactivated = mask_sale & (monthly_full['prev_churn'] == 1)
        monthly_full.loc[mask_reactivated, 'stage'] = 'Reactivated'
        mask_retained = mask_sale & monthly_full['stage'].isna()
        monthly_full.loc[mask_retained, 'stage'] = 'Retained'

        stage_order = ['New', 'Reactivated', 'Retained', 'Churn', 'Churn <= 3M', 'Churn > 3M']
        monthly_full['stage'] = monthly_full['stage'].astype(pd.CategoricalDtype(categories=stage_order, ordered=False))

        monthly_full.drop(columns=['first_positive_period','prev_churn'], inplace=True, errors='ignore')

        master = monthly_full.merge(stores, on='store_id', how='left', suffixes=('','_store'))

        return master

In [15]:
df_stores = pd.read_csv('data/raw/stores_data.csv')
df_orders = pd.read_csv('data/raw/orders_data.csv')
df_connectivity = pd.read_csv('data/raw/connectivity_data.csv')

In [16]:
df_orders.rename(columns={'created_date': 'orders_date',
                          'APP_ORIGIN': 'app_origin',
                          'promedio_shipping_cost': 'avg_shipping_cost',
                          'free_shipping_flag': 'free_shipping',
                          'TPN': 'tpn',
                          'TPV': 'tpv',
                          'GMV_LC': 'gmv'
                          }, inplace=True)

In [21]:
builder = MonthlyMasterBuilder()
stores_pre = StorePreprocessor().fit_transform(df_stores)
orders_pre = OrdersPreprocessor().fit_transform(df_orders)
monthly_master = builder.transform({'orders': orders_pre, 'stores': stores_pre})

/var/folders/3y/gthkm3q95_s8ph7_t5zg1wb00000gn/T/ipykernel_3036/1319116129.py:128: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.groupby('store_id', sort=False).apply(lambda g: g.bfill().iloc[0]).reset_index(drop=True)
/var/folders/3y/gthkm3q95_s8ph7_t5zg1wb00000gn/T/ipykernel_3036/1319116129.py:128: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('store_id', sort=False).apply(lambda g: g.bfill().iloc[0]).reset_index(drop=True)
/var/folders/3y/gthkm3q95_s8ph7

In [22]:
monthly_master.head(10)

,store_id,period_month,tpn,gmv,days_with_orders,markdown_mean,avg_shipping_cost,si_ios_prop,si_meli_prop,si_other_prop,...,segmento,comision_actual,in-house_delivery,store_type,barrio,category,cooking_time,tiene_otra_app,brand_id,ciudad
0,100000,2024-10,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
1,100000,2024-11,10.0,99900.0,4,0.192647,1122.625000,0.250000,0.000000,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
2,100000,2024-12,100.0,1494618.0,23,0.213178,1175.461111,0.444444,0.000000,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
3,100000,2025-01,78.0,1165196.0,20,0.395979,1168.689600,0.240000,0.120000,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
4,100000,2025-02,48.0,808800.0,15,0.400081,1171.896667,0.166667,0.111111,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
5,100000,2025-03,22.0,307838.0,9,0.355330,1141.647778,0.000000,0.000000,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
6,100000,2025-04,116.0,1665194.0,27,0.474851,1220.042195,0.243902,0.146341,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
7,100000,2025-05,10.0,121998.0,4,0.000000,1197.730000,0.250000,0.250000,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
8,100000,2025-06,12.0,201952.0,6,0.080861,1172.906667,0.333333,0.333333,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA
9,100000,2025-07,40.0,533694.0,12,0.342163,1154.513333,0.266667,0.266667,0.0,...,Top Players,19.5,0,Brand expansion,martinez oeste,Pollo,10,0,13736,GBA


In [23]:
monthly_master[monthly_master['ciudad'].isna()]['barrio'].value_counts()

barrio
nan                           10065
puerto deseado                   10
parque latino                     8
echesortu                         5
san andres                        5
ayacucho                          4
ameghino sud                      4
colinas de velez sarsfield        3
nueva cordoba anexa               3
alberto olmedo                    3
ona                               3
espana y hospitales               2
sarmiento                         2
Name: count, dtype: int64

In [24]:
monthly_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87111 entries, 0 to 87110
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   store_id                 87111 non-null  int64         
 1   period_month             87111 non-null  period[M]     
 2   tpn                      87111 non-null  float64       
 3   gmv                      87111 non-null  float64       
 4   days_with_orders         87111 non-null  int64         
 5   markdown_mean            38538 non-null  float64       
 6   avg_shipping_cost        38538 non-null  float64       
 7   si_ios_prop              38538 non-null  float64       
 8   si_meli_prop             38538 non-null  float64       
 9   si_other_prop            38538 non-null  float64       
 10  free_shipping_prop       38538 non-null  float64       
 11  churn                    54058 non-null  float64       
 12  churn_streak             54058 n

In [30]:
df_connectivity[df_connectivity['store_id'] == 112005].head()

,created_date,TOTAL_MIN_OPEN,TOTAL_MIN_PAUSED,TOTA_MIN_CLOSED,TOTAL_MINUTES_ALL,store_id
0,2024-11-19,88.1593,0.0000,11.8407,100.0,112005
6022,2025-01-04,92.4262,0.0000,7.5738,100.0,112005
7741,2025-08-05,49.4080,43.2723,7.3197,100.0,112005
8769,2024-12-16,95.0000,1.2963,3.7037,100.0,112005
8801,2024-12-26,82.8848,3.2293,13.8859,100.0,112005


In [33]:
monthly_master[monthly_master['store_id'] == 112005][['store_id', 'period_month', 'tpn', 'churn', 'churn_streak', 'churn_2m_consec', 'churn_4m_consec_or_more', 'stage', 'churn_unique']]

,store_id,period_month,tpn,churn,churn_streak,churn_2m_consec,churn_4m_consec_or_more,stage,churn_unique
66981,112005,2024-07,0.0,NaN,NaN,NaN,NaN,NaN,NaN
66982,112005,2024-08,58.0,0.0,0.0,0.0,0.0,New,0.0
66983,112005,2024-09,30.0,0.0,0.0,0.0,0.0,Retained,0.0
66984,112005,2024-10,30.0,0.0,0.0,0.0,0.0,Retained,0.0
66985,112005,2024-11,48.0,0.0,0.0,0.0,0.0,Retained,0.0
66986,112005,2024-12,36.0,0.0,0.0,0.0,0.0,Retained,0.0
66987,112005,2025-01,48.0,0.0,0.0,0.0,0.0,Retained,0.0
66988,112005,2025-02,86.0,0.0,0.0,0.0,0.0,Retained,0.0
66989,112005,2025-03,48.0,0.0,0.0,0.0,0.0,Retained,0.0
66990,112005,2025-04,40.0,0.0,0.0,0.0,0.0,Retained,0.0


In [32]:
monthly_master.to_csv('data/processed/monthly_master.csv', index=False)